# Feature Engineering: NSL-KDD

Notebook นี้นำข้อมูล cleaned จากขั้นตอนก่อนหน้ามาสร้าง features สำหรับโมเดลตรวจจับ network intrusion โดยแยก features และ targets เป็นคนละไฟล์เพื่อป้องกันการเผลอนำคำตอบเข้าโมเดล

ข้อมูลเข้า: output/NSL-KDD/*_clean.csv  
ผลลัพธ์: output/NSL-KDD/feature_engineered/

ลำดับงาน:

1. โหลด train และ test ที่ผ่าน cleansing แล้ว
2. แยก input features ออกจาก target
3. สร้าง log, indicator, ratio และ aggregate features
4. One-Hot Encode protocol_type, service, flag โดยยึดหมวดหมู่จาก train
5. ตรวจ schema และค่าของผลลัพธ์
6. บันทึก features, targets, schema และรายงาน

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. กำหนด path และโหลดข้อมูล cleaned

In [ ]:
CLEAN_DIR = Path("output/NSL-KDD")
OUTPUT_DIR = CLEAN_DIR / "feature_engineered"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILES = {
    "KDDTrain": CLEAN_DIR / "KDDTrain_clean.csv",
    "KDDTest": CLEAN_DIR / "KDDTest_clean.csv",
    "KDDTest-21": CLEAN_DIR / "KDDTest-21_clean.csv",
}

for dataset_name, path in DATA_FILES.items():
    if not path.exists():
        raise FileNotFoundError(
            f"ไม่พบ {dataset_name}: {path.resolve()} "
            "กรุณารัน nsl_cleansing.ipynb ก่อน"
        )

datasets = {name: pd.read_csv(path) for name, path in DATA_FILES.items()}

categorical_cols = ["protocol_type", "service", "flag"]
target_cols = ["label", "attack_family", "is_attack"]

for frame in datasets.values():
    for col in categorical_cols + ["label"]:
        frame[col] = frame[col].astype("string").str.strip().str.lower()
    frame["attack_family"] = frame["attack_family"].astype("string").str.strip()

summary = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(frame),
        "columns": frame.shape[1],
        "missing_cells": int(frame.isna().sum().sum()),
    }
    for name, frame in datasets.items()
])
display(summary)

## 2. แยก input features และ targets

label, attack_family และ is_attack เป็นคำตอบ (targets) ไม่ใช่ข้อมูล input ของโมเดล จึงแยกเก็บคนละไฟล์

difficulty และ num_outbound_cmds ควรถูกลบไปแล้วจาก cleansing notebook ถ้ายังพบคอลัมน์เหล่านี้จะหยุดเพื่อให้ตรวจ pipeline ก่อน

In [ ]:
train_columns = set(datasets["KDDTrain"].columns)
for dataset_name, frame in datasets.items():
    if set(frame.columns) != train_columns:
        raise ValueError(f"schema ของ {dataset_name} ไม่ตรงกับ KDDTrain")

for forbidden_col in ["difficulty", "num_outbound_cmds"]:
    if forbidden_col in train_columns:
        raise ValueError(
            f"ยังพบ {forbidden_col}; กรุณาตรวจ output จาก cleansing ก่อน"
        )

missing_targets = set(target_cols) - train_columns
if missing_targets:
    raise ValueError(f"ไม่พบ target columns: {sorted(missing_targets)}")

numeric_cols = [
    col for col in datasets["KDDTrain"].columns
    if col not in categorical_cols + target_cols
    and pd.api.types.is_numeric_dtype(datasets["KDDTrain"][col])
]

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric input columns: {len(numeric_cols)}")
print(f"Target columns: {target_cols}")

## 3. สร้าง feature จากการกระจายที่เบ้

byte และ count features มีหางยาว จึงสร้าง log1p(x) เพิ่มเพื่อบีบช่วงค่าขนาดใหญ่ โดยเก็บ feature ต้นฉบับไว้ด้วย

ตัวอย่าง: ค่า 0, 10 และ 1,000,000 จะถูกแปลงเป็นประมาณ 0, 2.40 และ 13.82 ทำให้ค่ามหาศาลไม่ครอบงำโมเดลที่ไวต่อสเกล

In [ ]:
log_cols = [
    "duration",
    "src_bytes",
    "dst_bytes",
    "count",
    "srv_count",
    "dst_host_count",
    "dst_host_srv_count",
    "hot",
    "num_failed_logins",
    "num_compromised",
    "num_root",
    "num_file_creations",
    "num_access_files",
]
log_cols = [col for col in log_cols if col in numeric_cols]

log_feature_names = ["log1p_" + col for col in log_cols]
print(f"สร้าง log1p features {len(log_feature_names)} ตัว:")
print(log_feature_names)

## 4. สร้าง indicator features สำหรับเหตุการณ์ที่พบไม่บ่อย

หลาย security features เป็นศูนย์เกือบทุกแถว เช่น failed login, root shell และ file creation ตัวบ่งชี้ 0/1 ช่วยให้โมเดลเห็นชัดว่าเหตุการณ์นั้นเกิดขึ้นหรือไม่

ตัวอย่าง: num_failed_logins = 0 ให้ has_num_failed_logins = 0; หากมี failed login อย่างน้อยหนึ่งครั้ง ให้เป็น 1

In [ ]:
indicator_cols = [
    "land",
    "wrong_fragment",
    "urgent",
    "hot",
    "num_failed_logins",
    "num_compromised",
    "root_shell",
    "su_attempted",
    "num_root",
    "num_file_creations",
    "num_shells",
    "num_access_files",
]
indicator_cols = [col for col in indicator_cols if col in numeric_cols]

indicator_feature_names = ["has_" + col for col in indicator_cols]
group_feature_names = ["has_root_activity", "has_file_or_shell_activity"]
print(f"สร้าง indicator features {len(indicator_feature_names) + len(group_feature_names)} ตัว")

## 5. สร้าง ratio, difference และ aggregate features

ฟีเจอร์ชุดนี้สรุปความสัมพันธ์ระหว่างค่าที่เกี่ยวข้อง เช่น traffic ไป-กลับ, จำนวน connection ต่อ service และ error rate ในระดับ host

เมื่อมีการหารจะบวก 1 ที่ตัวหารเพื่อป้องกันหารด้วยศูนย์ สำหรับผลต่างของ bytes ใช้ signed log เพื่อคงทิศทาง พร้อมลดผลของค่าขนาดใหญ่
หมายเหตุ: srv_to_connection_ratio และ host_service_count_ratio เป็นดัชนีเปรียบเทียบจากตัวนับคนละหน้าต่าง/กลุ่ม จึงอาจมีค่ามากกว่า 1 และไม่ควรตีความเป็น probability; connection_service_gap อาจติดลบได้ ทั้งหมดจะไม่ถูกบังคับให้อยู่ในช่วง 0–1.


In [ ]:
engineered_descriptions = {
    "total_bytes": "src_bytes + dst_bytes: ปริมาณข้อมูลรวมสองทิศทาง",
    "log1p_total_bytes": "log1p(total_bytes): ปริมาณรวมในสเกลที่บีบช่วงแล้ว",
    "signed_log_byte_difference": "sign(src-dst) * log1p(abs(src-dst)): ทิศทางและขนาดความต่างของ bytes",
    "log_byte_duration_ratio": "log1p(total_bytes) - log1p(duration): ความเข้มของ bytes เทียบกับ duration",
    "srv_to_connection_ratio": "srv_count / (count + 1): ดัชนีเปรียบเทียบ connection กับ service count",
    "connection_service_gap": "count - srv_count: ผลต่างระหว่าง connection count กับ service count; อาจติดลบ",
    "syn_error_mean": "ค่าเฉลี่ย serror_rate และ srv_serror_rate",
    "reject_error_mean": "ค่าเฉลี่ย rerror_rate และ srv_rerror_rate",
    "overall_error_rate": "ค่าเฉลี่ย error rate ระดับ connection/service",
    "host_service_count_ratio": "dst_host_srv_count / (dst_host_count + 1): ดัชนีเปรียบเทียบที่อาจมากกว่า 1",
    "host_service_rate_gap": "dst_host_same_srv_rate - same_srv_rate",
    "host_error_mean": "ค่าเฉลี่ย error rate สี่ตัวในระดับ destination host",
    "has_root_activity": "มี root_shell, su_attempted หรือ num_root มากกว่า 0",
    "has_file_or_shell_activity": "มี file creation, shell หรือ access file มากกว่า 0",
}

def build_engineered_columns(frame):
    result = pd.DataFrame(index=frame.index)

    for col in log_cols:
        result["log1p_" + col] = np.log1p(frame[col].clip(lower=0))

    for col in indicator_cols:
        result["has_" + col] = (frame[col] > 0).astype("int8")

    result["has_root_activity"] = (
        (frame["root_shell"] > 0)
        | (frame["su_attempted"] > 0)
        | (frame["num_root"] > 0)
    ).astype("int8")

    result["has_file_or_shell_activity"] = (
        (frame["num_file_creations"] > 0)
        | (frame["num_shells"] > 0)
        | (frame["num_access_files"] > 0)
    ).astype("int8")

    result["total_bytes"] = frame["src_bytes"] + frame["dst_bytes"]
    result["log1p_total_bytes"] = np.log1p(result["total_bytes"].clip(lower=0))

    byte_diff = frame["src_bytes"] - frame["dst_bytes"]
    result["signed_log_byte_difference"] = (
        np.sign(byte_diff) * np.log1p(byte_diff.abs())
    )
    result["log_byte_duration_ratio"] = (
        np.log1p(result["total_bytes"].clip(lower=0))
        - np.log1p(frame["duration"].clip(lower=0))
    )

    result["srv_to_connection_ratio"] = (
        frame["srv_count"] / (frame["count"] + 1)
    )
    result["connection_service_gap"] = frame["count"] - frame["srv_count"]

    result["syn_error_mean"] = (
        frame["serror_rate"] + frame["srv_serror_rate"]
    ) / 2
    result["reject_error_mean"] = (
        frame["rerror_rate"] + frame["srv_rerror_rate"]
    ) / 2
    result["overall_error_rate"] = frame[
        ["serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate"]
    ].mean(axis=1)

    result["host_service_count_ratio"] = (
        frame["dst_host_srv_count"] / (frame["dst_host_count"] + 1)
    )
    result["host_service_rate_gap"] = (
        frame["dst_host_same_srv_rate"] - frame["same_srv_rate"]
    )
    host_error_cols = [
        "dst_host_serror_rate",
        "dst_host_srv_serror_rate",
        "dst_host_rerror_rate",
        "dst_host_srv_rerror_rate",
    ]
    result["host_error_mean"] = frame[host_error_cols].mean(axis=1)

    return result

engineered_preview = build_engineered_columns(datasets["KDDTrain"])
display(engineered_preview.head())
print(f"เพิ่ม engineered numeric features: {engineered_preview.shape[1]} ตัว")

## 6. One-Hot Encode categorical features

protocol_type, service และ flag เป็นหมวดหมู่ จึงแปลงเป็นคอลัมน์ 0/1 เช่น protocol_type=tcp

ชุดหมวดหมู่และลำดับคอลัมน์อ้างอิงจาก train แล้วจัด test ให้มี schema เดียวกัน หาก test มีหมวดหมู่ที่ train ไม่เคยเห็น ค่า one-hot ของหมวดนั้นจะเป็นศูนย์ทุกคอลัมน์ในกลุ่มนั้น และจะถูกรายงานไว้

In [ ]:
train = datasets["KDDTrain"]
train_categories = {
    col: sorted(train[col].dropna().astype(str).unique().tolist())
    for col in categorical_cols
}

def encode_categories(frame):
    categorical = frame[categorical_cols].copy()
    for col in categorical_cols:
        categorical[col] = categorical[col].astype("string").fillna("__missing__")

    return pd.get_dummies(
        categorical,
        prefix=categorical_cols,
        prefix_sep="=",
        dtype="int8",
    )

train_ohe = encode_categories(train)
ohe_columns = train_ohe.columns.tolist()

unknown_category_rows = []
for dataset_name, frame in datasets.items():
    for col in categorical_cols:
        unseen = set(frame[col].dropna().astype(str).unique()) - set(train_categories[col])
        rows_with_unseen = int(frame[col].astype(str).isin(unseen).sum())
        unknown_category_rows.append({
            "dataset": dataset_name,
            "column": col,
            "unseen_categories": ", ".join(sorted(unseen)),
            "unseen_category_count": len(unseen),
            "rows_with_unseen_category": rows_with_unseen,
        })

unknown_category_report = pd.DataFrame(unknown_category_rows)
display(unknown_category_report)

## 7. ประกอบ feature matrix และแยก target

In [ ]:
def make_feature_matrix(frame):
    original_numeric = frame[numeric_cols].copy()
    engineered = build_engineered_columns(frame)
    categorical_encoded = encode_categories(frame).reindex(
        columns=ohe_columns,
        fill_value=0,
    )

    return pd.concat(
        [original_numeric, engineered, categorical_encoded],
        axis=1,
    )

feature_datasets = {}
target_datasets = {}

for dataset_name, frame in datasets.items():
    feature_datasets[dataset_name] = make_feature_matrix(frame)
    target_datasets[dataset_name] = frame[target_cols].copy()

train_feature_columns = feature_datasets["KDDTrain"].columns.tolist()
for dataset_name in feature_datasets:
    feature_datasets[dataset_name] = feature_datasets[dataset_name].reindex(
        columns=train_feature_columns,
        fill_value=0,
    )

shape_report = pd.DataFrame([
    {
        "dataset": name,
        "rows": len(feature_datasets[name]),
        "numeric_and_engineered_features": len(numeric_cols) + engineered_preview.shape[1],
        "one_hot_features": len(ohe_columns),
        "total_model_features": feature_datasets[name].shape[1],
        "target_columns": target_datasets[name].shape[1],
    }
    for name in feature_datasets
])
display(shape_report)

assert all(
    list(frame.columns) == train_feature_columns
    for frame in feature_datasets.values()
), "Train/test feature columns do not match"

assert all(
    len(feature_datasets[name]) == len(target_datasets[name])
    for name in feature_datasets
), "Features and targets row counts do not match"

## 8. ดูตัวอย่างก่อนและหลังสร้าง feature

In [ ]:
example = pd.concat(
    [
        datasets["KDDTrain"][[
            "src_bytes", "dst_bytes", "count", "srv_count",
            "serror_rate", "srv_serror_rate", "num_failed_logins",
        ]].head(5),
        engineered_preview[[
            "total_bytes", "log1p_src_bytes",
            "signed_log_byte_difference", "log_byte_duration_ratio",
            "srv_to_connection_ratio", "syn_error_mean",
            "has_num_failed_logins", "has_root_activity",
        ]].head(5),
    ],
    axis=1,
)
display(example)

## 9. ตรวจผลลัพธ์ก่อนบันทึก

In [ ]:
validation_rows = []

for dataset_name, feature_matrix in feature_datasets.items():
    target_frame = target_datasets[dataset_name]
    values = feature_matrix.to_numpy(dtype=float)

    validation_rows.append({
        "dataset": dataset_name,
        "feature_rows": len(feature_matrix),
        "target_rows": len(target_frame),
        "feature_columns": feature_matrix.shape[1],
        "missing_features": int(feature_matrix.isna().sum().sum()),
        "infinite_features": int(np.isinf(values).sum()),
        "target_missing": int(target_frame.isna().sum().sum()),
        "feature_schema_matches_train": (
            list(feature_matrix.columns) == train_feature_columns
        ),
    })

validation_report = pd.DataFrame(validation_rows)
display(validation_report)

if (
    validation_report["missing_features"].sum() > 0
    or validation_report["infinite_features"].sum() > 0
    or validation_report["target_missing"].sum() > 0
    or not validation_report["feature_schema_matches_train"].all()
):
    raise ValueError("Feature validation failed; inspect validation_report.")

## 10. บันทึก features, targets และ metadata

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for dataset_name in feature_datasets:
    feature_path = OUTPUT_DIR / f"{dataset_name}_features.csv"
    target_path = OUTPUT_DIR / f"{dataset_name}_targets.csv"

    feature_datasets[dataset_name].to_csv(feature_path, index=False)
    target_datasets[dataset_name].to_csv(target_path, index=False)

    print(f"Saved features: {feature_path.resolve()}")
    print(f"Saved targets:  {target_path.resolve()}")

feature_schema = pd.DataFrame({
    "feature": train_feature_columns,
    "kind": [
        (
            "one_hot"
            if col in ohe_columns
            else "engineered"
            if col in engineered_preview.columns
            else "original_numeric"
        )
        for col in train_feature_columns
    ],
})
feature_schema["description"] = feature_schema["feature"].map(
    lambda col: (
        engineered_descriptions.get(col, f"One-Hot category: {col}")
        if col in engineered_preview.columns or col in ohe_columns
        else f"Numeric feature from cleaned dataset: {col}"
    )
)
feature_schema.to_csv(OUTPUT_DIR / "feature_schema.csv", index=False)

unknown_category_report.to_csv(
    OUTPUT_DIR / "unknown_category_report.csv",
    index=False,
)
validation_report.to_csv(
    OUTPUT_DIR / "feature_validation_report.csv",
    index=False,
)

config = {
    "input_directory": str(CLEAN_DIR.resolve()),
    "output_directory": str(OUTPUT_DIR.resolve()),
    "categorical_columns": categorical_cols,
    "target_columns_separate_file": target_cols,
    "numeric_columns_from_cleaned_data": numeric_cols,
    "log1p_columns": log_cols,
    "indicator_columns": indicator_cols,
    "one_hot_categories_fitted_from": "KDDTrain only",
    "training_category_levels": train_categories,
    "feature_column_order": train_feature_columns,
    "unseen_test_category_policy": "all-zero one-hot vector; reported in unknown_category_report.csv",
    "scaling": "not applied; fit a scaler on training data inside the model pipeline when needed",
}
with (OUTPUT_DIR / "feature_engineering_config.json").open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(config, file, indent=2, ensure_ascii=False)

print("\nOutput files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

## 11. ตัวอย่างนำไปใช้กับโมเดล

ไฟล์ *_features.csv เป็น input ของโมเดล ส่วน *_targets.csv มี label, attack_family และ is_attack ให้เลือกเป็น target

ตัวอย่าง multiclass:

    X_train = pd.read_csv("output/NSL-KDD/feature_engineered/KDDTrain_features.csv")
    y_train = pd.read_csv("output/NSL-KDD/feature_engineered/KDDTrain_targets.csv")["attack_family"]

สำหรับ binary classification ให้ใช้คอลัมน์ is_attack แทน ควรแบ่ง validation จาก train ก่อนปรับโมเดล และใช้ KDDTest ประเมินรอบสุดท้าย